# Assignment 04 — 00: Baseline (Base Model Evaluation)
**Track 1 / Option A (SFT -> DPO)** | Model: `Qwen/Qwen3-0.6B-Base`

Group: **Abdullah Iqbal (26904), Anushe Ali (26418)**

This notebook records the *base* model's answers to the 10 manual prompts and
computes BLEU + BERTScore against the gold references. These numbers are the
baseline that SFT and DPO are compared against.

## 1. Install & mount

In [1]:
# Run once per Colab session. Restart runtime if prompted after install.
!pip install -q -U "transformers>=4.51" "trl>=0.15" "peft>=0.11" \
    "datasets>=2.19" accelerate sacrebleu bert-score matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJ = '/content/drive/MyDrive/assignment-4'   # change if you like
os.makedirs(PROJ + '/results', exist_ok=True)
os.makedirs(PROJ + '/adapters', exist_ok=True)
print('Saving outputs to', PROJ)

Mounted at /content/drive
Saving outputs to /content/drive/MyDrive/assignment-4


In [3]:
import json, torch
PROMPT_TEMPLATE = '### Instruction:\n{instruction}\n\n### Response:\n'
def format_prompt(instruction):
    return PROMPT_TEMPLATE.format(instruction=instruction.strip())
def pick_dtype():
    if torch.cuda.is_available():
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.float32

@torch.no_grad()
def generate_response(model, tokenizer, instruction, max_new_tokens=256):
    prompt = format_prompt(instruction)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    gen = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def generate_all(model, tokenizer, test_set, max_new_tokens=256):
    rows = []
    for ex in test_set:
        rows.append({'id': ex['id'], 'instruction': ex['instruction'],
                     'reference': ex['reference'],
                     'response': generate_response(model, tokenizer, ex['instruction'], max_new_tokens)})
    return rows

def compute_bleu(hyps, refs):
    import sacrebleu
    s = [sacrebleu.sentence_bleu(h, [r]).score for h, r in zip(hyps, refs)]
    return sum(s) / max(len(s), 1)

def compute_bertscore(hyps, refs, model_type='roberta-large'):
    from bert_score import score as bert_score
    P, R, F1 = bert_score(hyps, refs, lang='en', model_type=model_type, verbose=False)
    return float(F1.mean())

def evaluate_rows(rows, bertscore_model='roberta-large'):
    hyps = [r['response'] for r in rows]; refs = [r['reference'] for r in rows]
    bleu = compute_bleu(hyps, refs); bert = compute_bertscore(hyps, refs, bertscore_model)
    return {'bleu': bleu, 'bertscore_f1': bert, 'composite': 0.5*(bleu/100.0)+0.5*bert}

def select_best(trials, tol=0.005):
    ranked = sorted(trials, key=lambda t: t['composite'], reverse=True)
    top = ranked[0]['composite']
    cont = [t for t in ranked if top - t['composite'] <= tol]
    if len(cont) > 1:
        cont = sorted(cont, key=lambda t: t.get('val_loss', float('inf')))
    return cont[0]

In [5]:
# Upload data/test_set.json to PROJ (or to Colab and adjust path).
# IMPORTANT: fill the 'reference' fields with gold answers from ChatGPT/Claude/Gemini first!
TEST_PATH = PROJ + '/data/test_set.json'
with open(TEST_PATH) as f:
    test_set = json.load(f)
assert all('<<PASTE' not in ex['reference'] for ex in test_set), \
    'Fill in the gold reference answers in test_set.json before running!'
print(len(test_set), 'test prompts loaded')

10 test prompts loaded


## 2. Load base model

In [6]:
MODEL_ID = 'Qwen/Qwen3-0.6B-Base'
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=pick_dtype(), device_map='auto')
base_model.eval()
print('Loaded', MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Loaded Qwen/Qwen3-0.6B-Base


## 3. Generate baseline responses

In [7]:
baseline_rows = generate_all(base_model, tokenizer, test_set)
for r in baseline_rows[:2]:
    print('Q:', r['instruction']); print('A:', r['response'][:300]); print('-'*60)

Q: Explain the difference between supervised and unsupervised learning in simple terms.
A: Supervised learning and unsupervised learning are two types of machine learning algorithms used to train models on data. In supervised learning, the model is trained on labeled data, meaning that each input is paired with a corresponding output. The goal is to learn a mapping from inputs to outputs 
------------------------------------------------------------
Q: Write a short, polite email to a professor requesting a one-week extension on an assignment.
A: Write a short, polite email to a professor requesting a one-week extension on an assignment.

### Instructions:
- Use a professional and polite tone.
- Clearly state the reason for the request.
- Provide any relevant information or context.
- Thank the professor for their time and consideration.
- E
------------------------------------------------------------


## 4. Compute BLEU + BERTScore

In [8]:
import pandas as pd
baseline_metrics = evaluate_rows(baseline_rows)
print('BASELINE  BLEU = %.2f   BERTScore-F1 = %.4f   composite = %.4f' %
      (baseline_metrics['bleu'], baseline_metrics['bertscore_f1'], baseline_metrics['composite']))
pd.DataFrame(baseline_rows)[['id','instruction','response']]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BASELINE  BLEU = 9.82   BERTScore-F1 = 0.8803   composite = 0.4892


,id,instruction,response
0,1,Explain the difference between supervised and ...,Supervised learning and unsupervised learning ...
1,2,"Write a short, polite email to a professor req...","Write a short, polite email to a professor req..."
2,3,What are three practical tips for improving fo...,Certainly! Here are three practical tips for i...
3,4,Summarize the water cycle in two sentences.,The water cycle is a continuous process that i...
4,5,Translate the sentence 'Knowledge is power' in...,"Sure, here are the translations:\n\n- **French..."
5,6,Give me a simple recipe for making pancakes fr...,Sure! Here's a simple recipe for making pancak...
6,7,What is photosynthesis and why is it important...,Photosynthesis is the process by which green p...
7,8,List the steps to set up a basic Python virtua...,1. Create a new directory for your virtual env...
8,9,Explain what a stock market index is to someon...,A stock market index is a measure of the perfo...
9,10,Describe the benefits of regular physical exer...,Regular physical exercise offers numerous bene...


## 5. Save baseline results

In [9]:
out = {'stage': 'baseline', 'model': MODEL_ID,
       'metrics': baseline_metrics, 'rows': baseline_rows}
with open(PROJ + '/results/baseline.json', 'w') as f:
    json.dump(out, f, indent=2)
print('Saved baseline.json')

Saved baseline.json


### Notes for the report
- Record platform (Colab GPU type via `!nvidia-smi`), model + tokenizer choice.
- The base model is *not* instruction-tuned, so expect low BLEU and rambling answers.
- This is intentional: it makes the SFT improvement clearly visible.

In [10]:
!nvidia-smi

Sat May 30 00:55:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P0             31W /   70W |    2655MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----